<a href="https://colab.research.google.com/github/MelisaYasak/transformer_arch/blob/main/sherlockGPT_biagram_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content/drive/MyDrive/transformers

/content/drive/MyDrive/transformers


# Bigram Modeli

Bigram bir sonraki token'ı tahmin etmek içi sadece o stepdeki token'ı dikkate alan basic bir dil modeli.

**_P(x_t+1 | x_t)_**

Dipnot: Bağlamı sadece bir karakterdir.

## Data Prep

In [2]:
from pathlib import Path

In [4]:
text = Path("sherlock.txt").read_text(
    encoding="utf-8-sig"
)

In [5]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

In [6]:
start = text.find(start_marker)
end = text.find(end_marker)

In [7]:
if start != -1:
    text = text[text.find("\n", start) + 1:]

if end != -1:
    text = text[:text.find(end_marker)]

In [8]:
text = text.strip()

In [9]:
print("Karakter sayısı:", len(text))
print(text[:300])

Karakter sayısı: 562202
The Adventures of Sherlock Holmes

by Arthur Conan Doyle


Contents

   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the 


# Tokenizer ve Dataset

### Dataset oluştur

In [10]:
import torch

In [11]:
torch.manual_seed(42)

In [12]:
chars = sorted(set(text))

In [13]:
vocab_size = len(chars)
vocab_size

88

In [14]:
stoi = {ch: i for i, ch in enumerate(chars)}

itos = {i: ch for ch, i in stoi.items()}

In [15]:
encode = lambda s: [stoi[c] for c in s]

decode = lambda ids: "".join(itos[i] for i in ids)

In [16]:
data = torch.tensor(encode(text), dtype=torch.long)

In [17]:
# eğitimden önce veriyi train ve test olarak ayıracağız

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [18]:
print("Vocabulary size:", vocab_size)
print("Train shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("İlk karakterler:", decode(data[:50].tolist()))

Vocabulary size: 88
Train shape: torch.Size([505981])
Validation shape: torch.Size([56221])
İlk karakterler: The Adventures of Sherlock Holmes

by Arthur Conan


### Modeli Kur

In [19]:
import torch.nn as nn
import torch.nn.functional as F

In [20]:
class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    # Bir embedding oluşturacağız.
    # Tabloda vocab_size kadar farklı token bulunacak
    # ve her token vocab_size kadar boyutlu vektöre dönüştürülecek.
    # Bigram modelinde her karakter için vocabulary boyutunda bir logit vektörü kullanıyoruz.
    self.token_embedding_table = nn.Embedding(
        vocab_size,
        vocab_size
    )

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)

    loss = None

    if targets is not None:
      B, T, C = logits.shape

      logits_flat = logits.reshape(B * T, C)
      targets_flat = targets.reshape(B * T)

      loss = F.cross_entropy(logits_flat, targets_flat)

    return logits, loss

In [21]:
batch_size = 16
block_size = 8

In [22]:
def get_batch(split):
  source = train_data if split == "train" else val_data

  ix = torch.randint(
      0,
      len(source) - block_size,
      (batch_size,)
  )

  x = torch.stack([
      source[i:i + block_size]
      for i in ix
  ])

  y = torch.stack([
      source[i+1:i + block_size + 1]
      for i in ix
  ])

  return x, y

xb, yb = get_batch("train")

In [23]:
print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("Input:", decode(xb[0].tolist()))
print("Target:", decode(yb[0].tolist()))

Input shape: torch.Size([16, 8])
Target shape: torch.Size([16, 8])
Input: : It was
Target:  It was.


### Eğitime Başla

In [24]:
model = BigramLanguageModel(vocab_size)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("Başlangıç loss:", loss.item())

Başlangıç loss: 4.953321933746338


In [30]:
for step in range(10000):
  xb, yb = get_batch("train")
  logits, loss = model(xb, yb)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if step % 200 == 0:
    print(
            f"Step {step} | "
            f"Train batch loss: {loss.item():.4f}"
        )

Step 0 | Train batch loss: 2.2412
Step 200 | Train batch loss: 2.4165
Step 400 | Train batch loss: 2.3734
Step 600 | Train batch loss: 2.5783
Step 800 | Train batch loss: 2.3773
Step 1000 | Train batch loss: 2.4398
Step 1200 | Train batch loss: 2.4217
Step 1400 | Train batch loss: 2.4691
Step 1600 | Train batch loss: 2.3925
Step 1800 | Train batch loss: 2.4252
Step 2000 | Train batch loss: 2.4599
Step 2200 | Train batch loss: 2.2989
Step 2400 | Train batch loss: 2.3994
Step 2600 | Train batch loss: 2.5739
Step 2800 | Train batch loss: 2.4793
Step 3000 | Train batch loss: 2.5253
Step 3200 | Train batch loss: 2.2914
Step 3400 | Train batch loss: 2.4539
Step 3600 | Train batch loss: 2.5558
Step 3800 | Train batch loss: 2.3649
Step 4000 | Train batch loss: 2.4232
Step 4200 | Train batch loss: 2.2697
Step 4400 | Train batch loss: 2.6540
Step 4600 | Train batch loss: 2.1767
Step 4800 | Train batch loss: 2.4118
Step 5000 | Train batch loss: 2.5459
Step 5200 | Train batch loss: 2.5055
Step 540

In [31]:
model.eval()

with torch.no_grad():
  xb, yb = get_batch("val")

  logits, val_loss = model(xb, yb)

print("Validation batch loss:", val_loss.item())


Validation batch loss: 2.4500973224639893


### Let's Go to the Test

In [33]:
@torch.no_grad()
def generate(model, start_id, max_new_tokens=200):
  model.eval()

  idx = torch.tensor([[start_id]], dtype=torch.long)

  for _ in range(max_new_tokens):

    logits, _ = model(idx)

    last_logits = logits[:, -1, :]

    probs = F.softmax(last_logits, dim=-1)

    next_id = torch.multinomial(
        probs,
        num_samples=1
    )

    idx = torch.cat([idx, next_id], dim=1)

  return decode(idx[0].tolist())

print(generate(model, stoi["S"]))

Sulol utouris incand a tawethiot anerenet at th owsotind hed f I oulenitoliurowindo an, tot g wentinllinlly blat onclk,”
meshontht, hingheintr meves pp
“On I cl oowind w d r beerald &xcernghim dsuthe w
